In [13]:
from simurg_plotter.manager import PlotManager, Plots
from simurg_plotter.ionospheric_pierce_point import Series

In [14]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [15]:
from pathlib import Path
from numpy.typing import NDArray
import datetime
import h5py
from dateutil import tz
TIME_FORMAT = '%Y-%m-%d %H:%M:%S.%f'
_UTC = tz.gettz('UTC')
def retrieve_data(
    file: str | Path,
    times: list[datetime.datetime] | None = None
) -> dict[datetime.datetime, NDArray]:    
    """
    Retrieves data from HDF file and put in dictionary
    Keys are datetime.datetime values are structured numpy arrays
    file - path to file
    times - times to preserve in output
    """  
    if times is None:
        times = []
    f_in = h5py.File(file, 'r')
    lats = []
    lons = []
    values = []
    data = {}
    for str_time in list(f_in['data'])[:]:
        time = datetime.datetime.strptime(str_time, TIME_FORMAT)
        time = time.replace(tzinfo=time.tzinfo or _UTC)
        if times and not time in times:
            # print(type(time))
            continue
        
        data[time] = f_in['data'][str_time][:]
    return data

In [16]:
times = [(datetime.datetime(2024, 4, 19) + datetime.timedelta(minutes=i)).replace(tzinfo=(datetime.datetime(2024, 4, 19) + datetime.timedelta(minutes=i)).tzinfo or _UTC) for i in range(30)]
data = retrieve_data("files/dtec_2_10_2024_110_50_56_N_100_110_E_b4aa.h5", times=times)

In [17]:
plotter = PlotManager(plots=[Plots.MAP2D])
plot_data = {Plots.MAP2D:(data, {"title": "Map2D", "product_type": "dtec_2_10", "time":None, "save_fig":None, "polar":False, "subsolar":True, "min_lat":50, "max_lat":55, "min_lon":100, "max_lon":110})} 
plotter.animate_plots(plot_data, times, file_name="map2d_animation.gif", fig_path="files/",fps=15)

In [18]:
from IPython.display import display, HTML
gif_path = 'files/map2d_animation.gif'
html_code = f'<img src="{gif_path}" autoplay loop>'
display(HTML(html_code))

In [19]:
import numpy as np

start_date = datetime.datetime(2024, 4, 19)
time_interval = datetime.timedelta(minutes=1)
timestamps = [start_date + i * time_interval for i in range(60)]
dst_values = np.random.uniform(low=-100, high=100, size=len(timestamps))
dst_data = np.column_stack((timestamps, dst_values))

In [20]:
times = [(datetime.datetime(2024, 4, 19) + datetime.timedelta(minutes=i)).replace(tzinfo=(datetime.datetime(2024, 4, 19) + datetime.timedelta(minutes=i)).tzinfo or _UTC) for i in range(30)]

In [21]:
plotter = PlotManager(plots=[Plots.MAP2D, Plots.DST])
plot_data = {Plots.MAP2D:(data, {"title": "Map2D", "product_type": "dtec_2_10", "time":None, "save_fig":None, "polar":False, "subsolar":True, "min_lat":50, "max_lat":55, "min_lon":100, "max_lon":110}),
            Plots.DST:(dst_data, {"time":None})
            } 
plotter.animate_plots(plot_data, times, file_name="map2d_dst_animation.gif", fig_path="files/",fps=15)

In [22]:
from IPython.display import display, HTML
gif_path = 'files/map2d_dst_animation.gif'
html_code = f'<img src="{gif_path}" autoplay loop>'
display(HTML(html_code))